# Respiratory CDSS XGBoost Training

Bu notebook diplom uchun model comparison bosqichini tayyorlaydi. Hozirgi loyiha baseline sifatida pure-Python Naive Bayes ishlatadi; shu notebook esa keyingi bosqichda XGBoost modelini sinash uchun skelet beradi.

Muhim eslatma:
- agar `xgboost` lokal muhitda o'rnatilmagan bo'lsa, notebook trainingni skip qiladi
- feature dataset va split manifest mavjud bo'lsa, train/test comparison tayyor bo'ladi
- bu notebook real dataset bilan qayta ishlatish uchun yozilgan


In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path
from pprint import pprint


def find_backend_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "app").exists() and (candidate / "data").exists():
            return candidate
        backend_candidate = candidate / "backend"
        if (backend_candidate / "app").exists() and (backend_candidate / "data").exists():
            return backend_candidate
    raise RuntimeError("Backend root topilmadi")


BACKEND_ROOT = find_backend_root(Path.cwd())
DATA_DIR = BACKEND_ROOT / "data"
MODEL_DIR = BACKEND_ROOT / "ml_models"

FEATURE_DATASET_PATH = DATA_DIR / "respiratory_feature_dataset.csv"
SPLIT_PATH = DATA_DIR / "respiratory_train_test_split.json"
NB_METRICS_PATH = MODEL_DIR / "respiratory_nb_metrics.json"
NB_EVALUATION_PATH = MODEL_DIR / "respiratory_nb_evaluation.json"

print(f"Backend root: {BACKEND_ROOT}")
print(f"Feature dataset: {FEATURE_DATASET_PATH}")


In [ ]:
def load_csv_rows(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8", newline="") as csv_file:
        return list(csv.DictReader(csv_file))


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as json_file:
        return json.load(json_file)


feature_rows = load_csv_rows(FEATURE_DATASET_PATH)
split_manifest = load_json(SPLIT_PATH)
nb_metrics = load_json(NB_METRICS_PATH)
nb_evaluation = load_json(NB_EVALUATION_PATH)

print(f"Feature rows loaded: {len(feature_rows)}")
print(f"Train IDs: {len(split_manifest['train_ids'])} | Test IDs: {len(split_manifest['test_ids'])}")


In [ ]:
feature_names = [column for column in feature_rows[0].keys() if column not in {"sample_id", "label"}]
label_names = sorted({row['label'] for row in feature_rows})
label_to_index = {label: index for index, label in enumerate(label_names)}


def encode_feature_rows(rows: list[dict[str, str]], columns: list[str], value_maps=None, fit=False):
    if value_maps is None:
        value_maps = {column: {} for column in columns}
        fit = True
    encoded_rows = []

    for row in rows:
        encoded_row = []
        for column in columns:
            value = row[column]
            mapping = value_maps[column]
            if value not in mapping:
                if fit:
                    mapping[value] = len(mapping)
                else:
                    encoded_row.append(-1)
                    continue
            encoded_row.append(mapping[value])
        encoded_rows.append(encoded_row)

    return encoded_rows, value_maps


row_by_id = {row['sample_id']: row for row in feature_rows}

print(f"Encoded feature count: {len(feature_names)}")
print(f"Label set: {label_names}")


In [ ]:
train_rows = [row_by_id[sample_id] for sample_id in split_manifest['train_ids'] if sample_id in row_by_id]
test_rows = [row_by_id[sample_id] for sample_id in split_manifest['test_ids'] if sample_id in row_by_id]

train_encoded, feature_value_maps = encode_feature_rows(train_rows, feature_names)
test_encoded, _ = encode_feature_rows(test_rows, feature_names, value_maps=feature_value_maps, fit=False)

y_train = [label_to_index[row['label']] for row in train_rows]
y_test = [label_to_index[row['label']] for row in test_rows]

print(f"Train rows: {len(train_rows)}")
print(f"Test rows: {len(test_rows)}")


In [ ]:
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print(f"xgboost imported successfully: {xgb.__version__}")
except ImportError:
    xgb = None
    XGBOOST_AVAILABLE = False
    print("xgboost topilmadi. Bu notebook training qismini skip qiladi.")
    print("Agar keyin kerak bo'lsa, lokal muhitga xgboost o'rnatib notebookni qayta ishga tushiring.")


In [ ]:
print("Naive Bayes baseline summary:")
print(f"- Holdout accuracy: {nb_metrics['metrics']['accuracy']}")
print(f"- Train samples: {nb_metrics['train_samples']}")
print(f"- Test samples: {nb_metrics['test_samples']}")
print(f"- CV overall accuracy: {nb_evaluation['overall_accuracy']}")

print("\nPer-label CV accuracy:")
for label, score in nb_evaluation['per_label_accuracy'].items():
    print(f"- {label}: {score}")


In [ ]:
xgb_results = None

if XGBOOST_AVAILABLE:
    model = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_names),
        n_estimators=80,
        max_depth=4,
        learning_rate=0.1,
        subsample=1.0,
        colsample_bytree=1.0,
        eval_metric="mlogloss",
        random_state=42,
    )
    model.fit(train_encoded, y_train)
    predictions = model.predict(test_encoded)
    correct = sum(int(pred == expected) for pred, expected in zip(predictions, y_test))
    accuracy = round(correct / len(y_test), 3) if y_test else 1.0

    feature_importances = sorted(
        zip(feature_names, model.feature_importances_),
        key=lambda item: item[1],
        reverse=True,
    )
    xgb_results = {
        "holdout_accuracy": accuracy,
        "top_feature_importances": feature_importances[:10],
    }
    pprint(xgb_results)
else:
    print("Training skipped: xgboost dependency mavjud emas.")


In [ ]:
print("Model comparison summary:")
print(f"- Naive Bayes holdout accuracy: {nb_metrics['metrics']['accuracy']}")
print(f"- Naive Bayes CV accuracy: {nb_evaluation['overall_accuracy']}")

if xgb_results is not None:
    print(f"- XGBoost holdout accuracy: {xgb_results['holdout_accuracy']}")
    print("\nTop XGBoost feature importances:")
    for feature_name, importance in xgb_results['top_feature_importances']:
        print(f"- {feature_name}: {round(float(importance), 4)}")
else:
    print("- XGBoost natijasi hozircha yo'q (dependency missing yoki training skip qilindi)")


## Diplom uchun ishlatish tavsiyasi

1. Avval real datasetni onboarding + cleaning + pipeline bosqichlaridan o'tkazing.
2. Shu notebookda NB va XGBoost natijalarini bir jadvalda solishtiring.
3. Agar XGBoost ishlatilsa, feature importance natijalarini interpretatsiya bo'limiga olib chiqing.
4. Seed dataset natijalarini ilmiy yakuniy xulosa sifatida emas, texnik prototip sifatida ko'rsating.
